__Actividad Taller Integrador - Consumo de APIs__

API: https://dogapi.dog/docs/api-v2 

Descripción: esta API proporciona información detallada sobre el mundo canino. Su objetivo es centralizar datos precisos sobre razas de perros, sus características físicas, grupos taxonómicos y estados de salud.

Detalles: esta API no requiere llaves para consultas básicas. 

In [5]:
# Importamos las librerias
import requests
import pandas as pd

In [6]:
# Consultamos la API
url_base = "https://dogapi.dog/api/v2"
endopoint = "/breeds"
response = requests.get(url_base + endopoint)

url_final = f"{url_base}{endopoint}"
print(url_final)

https://dogapi.dog/api/v2/breeds


In [28]:
# Verificamos el estado de conexión con la API
if response.status_code == 200:
    print('Conexión exitosa a la API. Código de estado:', response.status_code)
else:
    print('Conexión fallida a la API. Código de estado:', response.status_code)

Conexión exitosa a la API. Código de estado: 200


In [29]:
# Leemos los datos en json
datos_completos = response.json()
datos_json = datos_completos.get('data', [])

print(datos_json[0])

{'id': '036feed0-da8a-42c9-ab9a-57449b530b13', 'type': 'breed', 'attributes': {'name': 'Affenpinscher', 'description': 'The Affenpinscher is a small and playful breed of dog that was originally bred in Germany for hunting small game. They are intelligent, energetic, and affectionate, and make excellent companion dogs.', 'life': {'max': 16, 'min': 14}, 'male_weight': {'max': 5, 'min': 3}, 'female_weight': {'max': 5, 'min': 3}, 'hypoallergenic': True}, 'relationships': {'group': {'data': {'id': 'f56dc4b1-ba1a-4454-8ce2-bd5d41404a0c', 'type': 'group'}}}}


In [ ]:
# Organizamos los datos en columnas y se hacen un cálculos adicionales con el fin de que el Dasboard en Power BI entregue información práctica basada en la
# caraterización inicial que se recopila en la API.
tabla_final = []

for item in datos_json:
    attrs = item.get('attributes', {})

    perro = {
        'Raza': attrs.get('name'),
        'Hipoalergenico': 'Sí' if attrs.get('hypoallergenic') else 'No',
        'Vida_Min': attrs.get('life', {}).get('min'),
        'Vida_Max': attrs.get('life', {}).get('max'),
        'Peso_Macho': attrs.get('male_weight', {}).get('max'),
        'Peso_Hembra': attrs.get('female_weight', {}).get('max'),
        'Descripcion': attrs.get('description')
    }

# Cálculos adicionales
    perro['Vida_Promedio'] = (perro['Vida_Min'] + perro['Vida_Max']) / 2
    perro['Peso_Promedio'] = (perro['Peso_Macho'] + perro['Peso_Hembra']) / 2
    
# Incluyo una categorización de Longevidad (duración de vida)
    v = perro['Vida_Promedio']
    if v <= 10:perro['Longevidad'] = 'Corta'
    elif v <= 13:perro['Longevidad'] = 'Media'
    else:perro['Longevidad'] = 'Alta'

# Incluyo una categorización de tamaño 
    p = perro['Peso_Promedio']
    if p <= 10:perro['Tamaño'] = 'Pequeño'
    elif p <= 25:perro['Tamaño'] = 'Mediano'
    elif p <= 45:perro['Tamaño'] = 'Grande'
    else:perro['Tamaño'] = 'Gigante'         

# Incluyo una categorización de aptitud 
    desc = str(perro['Descripcion']).lower()
    if any(word in desc for word in ['hunt', 'hunter']): perro['Aptitud'] = 'Cazador'
    elif any(word in desc for word in ['guard', 'watchdog', 'protect']): perro['Aptitud'] = 'Guardián'
    elif any(word in desc for word in ['companion', 'family', 'pet']): perro['Aptitud'] = 'Compañía'
    else: perro['Aptitud'] = 'General'

    tabla_final.append(perro)

In [ ]:
# Se crea un dataframe y visualizamos como queda la tabla
df_perros = pd.DataFrame(tabla_final)
display(df_perros.head())

,Raza,Hipoalergenico,Vida_Min,Vida_Max,Peso_Macho,Peso_Hembra,Descripcion,Vida_Promedio,Peso_Promedio,Longevidad,Tamaño,Aptitud
0,Affenpinscher,Sí,14,16,5,5,The Affenpinscher is a small and playful breed...,15.0,5.0,Alta,Pequeño,Cazador
1,Afghan Hound,No,12,14,27,25,The Afghan Hound is a large and elegant breed ...,13.0,26.0,Media,Grande,Cazador
2,Airedale Terrier,No,12,14,23,20,The Airedale Terrier is a large and powerful b...,13.0,21.5,Media,Mediano,Cazador
3,Akita,No,10,12,60,50,"The Akita is a large, muscular dog breed that ...",11.0,55.0,Media,Gigante,General
4,Alaskan Klee Kai,No,12,15,7,7,The Alaskan Klee Kai is a small to medium-size...,13.5,7.0,Alta,Pequeño,General


In [34]:
# Exportamos el archivo para construir el Dasboard, sin indice
df_perros.to_csv("Taller_Api_Dogs.csv", index=False, encoding="utf-8-sig")